# 46 — Lemma-policy control: English-only Llama on the validated pool

The cross-family comparison (notebook 43) builds Llama concepts from 8
WordNet languages (min_lemmas=11) but Gemma concepts English-only
(min_lemmas=4) — the paper's per-family defaults. The same synset therefore
yields a different concept frame per family: a confound on the family-gap
claim (report §5.8, §3.4).

**Design.** Rerun notebook 43's Llama-bf16 arm with **Gemma's lemma policy**
(English-only, min_lemmas=4). Same 30 pairs (`resources/43_pairs.json`),
same three methods, prompts, and decoding (greedy, k=4, steps=24, λ=0).
Reading: if English-only Llama moves materially toward Gemma's ~99%
both-improved, the family gap is (partly) a lemma-policy artifact; if it
stays near the 8-language arm's rates, the gap is attributable to the model
family and the §5.8 caveat can be retired.

Concepts that do not survive the English-only policy are reported and their
pairs dropped, exactly as in 43; the comparison cell restricts to shared
(pair × method) cells for a paired read.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.nlp import SupportedLanguages
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin
from frames.utils.memory import gc_cuda

In [ ]:
MAX_TOKENS = 3
K = 4
STEPS = 24

# Gemma's policy, applied to Llama: English-only, min_lemmas=4 (43 used 11/8-lang)
MIN_LEMMAS = 4
MODEL_KEY = "llama-3.1-8b-bf16-eng"

PAIRS = json.loads(Path("resources/43_pairs.json").read_text())["pairs"]
SYNSETS = sorted({s for p in PAIRS for s in (p["a"], p["b"])})

LOG_PATH = Path("resources/46_lemma_policy.jsonl")
LOG_PATH.unlink(missing_ok=True)


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

In [ ]:
with gc_cuda():
    fur = FrameUnembeddingRepresentation.from_model_id(
        id="meta-llama/Llama-3.1-8B-Instruct",
        language_codes=SupportedLanguages.Default,  # English-only
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    prompts = [chat(u, a) for u, a in QUESTIONS]
    harness = EvaluationHarness(
        fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
    )
    baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

    concept, missing = {}, []
    for name in SYNSETS:
        try:
            c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
            concept[name] = Concept(
                synset=c.synset, tensor=c.tensor.to(fur.model.device)
            )
        except Exception as error:  # noqa: BLE001 - survival is data-dependent
            missing.append(name)
            print(f"  DROPPED {name}: {type(error).__name__}")
    pairs = [p for p in PAIRS if p["a"] not in missing and p["b"] not in missing]
    print(f"[{MODEL_KEY}] {len(concept)}/{len(SYNSETS)} concepts, {len(pairs)} pairs")

    for p in pairs:
        a, b = p["a"], p["b"]
        ca, cb = concept[a], concept[b]
        rho_model = float(ca.rho(cb))
        methods = {
            "blend": lambda: fur.generate_with_topk_guide(
                prompts, guide=Concept.average([ca, cb]), k=K, steps=STEPS
            ),
            "vote": lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=[ca, cb], k=K, steps=STEPS, normalize="zscore"
            ),
            "schedule": lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=[ca, cb], k=K, steps=STEPS, scorer=RoundRobin()
            ),
        }
        for method, generate in methods.items():
            torch.cuda.empty_cache()
            texts, probe = generate()
            harness.evaluate(
                prompts,
                texts,
                [a, b],
                config={
                    "model": MODEL_KEY,
                    "method": method,
                    "a": a,
                    "b": b,
                    "rho": rho_model,
                    "rho_pool": p["rho"],
                },
                probe=probe,
                baseline_texts=baseline_texts,
                concepts={a: ca, b: cb},
                angles=True,
            )
    print(f"[{MODEL_KEY}] done")

In [ ]:
def rows_from(path, model_filter=None):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        if model_filter and c["model"] != model_filter:
            continue
        da = r["expression"][c["a"]]["delta"]
        db = r["expression"][c["b"]]["delta"]
        ca = r["angles"][c["a"]]["delta"]["token_cos"]
        cb = r["angles"][c["b"]]["delta"]["token_cos"]
        rows.append(
            {
                "model": c["model"],
                "method": c["method"],
                "pair": f"{c['a']}|{c['b']}",
                "joint": min(da, db),
                "both_improved": min(da, db) > 0,
                "both_improved_cos": min(ca, cb) > 0,
                "ppl_ratio": r["ppl_ratio"],
            }
        )
    return pd.DataFrame(rows)


eng = rows_from(LOG_PATH)
multi = rows_from("resources/43_validated_pool.jsonl", "llama-3.1-8b-bf16")

# paired read: shared (pair x method) cells only
shared = set(eng[["method", "pair"]].apply(tuple, axis=1)) & set(
    multi[["method", "pair"]].apply(tuple, axis=1)
)
eng_s = eng[eng[["method", "pair"]].apply(tuple, axis=1).isin(shared)]
multi_s = multi[multi[["method", "pair"]].apply(tuple, axis=1).isin(shared)]

summary = pd.concat(
    [
        eng_s.groupby("method")
        .agg(
            both_improved=("both_improved", "mean"),
            both_improved_cos=("both_improved_cos", "mean"),
            joint=("joint", "mean"),
            ppl_ratio=("ppl_ratio", "mean"),
            n=("joint", "size"),
        )
        .assign(policy="english-only"),
        multi_s.groupby("method")
        .agg(
            both_improved=("both_improved", "mean"),
            both_improved_cos=("both_improved_cos", "mean"),
            joint=("joint", "mean"),
            ppl_ratio=("ppl_ratio", "mean"),
            n=("joint", "size"),
        )
        .assign(policy="8-language (nb43)"),
    ]
).reset_index()
summary.to_csv("resources/46_lemma_policy_summary.csv", index=False)
print(summary.round(3).to_string(index=False))
print(
    "\nGemma reference (nb43, English-only by default): "
    "gemma-2-2b ~99%, gemma-2-9b ~99% both-improved.\n"
    "Reading: english-only ≈ 8-language -> gap is a family property "
    "(retire the caveat); english-only >> 8-language -> lemma-policy "
    "artifact (revise §5.8)."
)